# Laboratoire SIG – SQL et statistiques (version enseignant corrigée)

## 1. Introduction et objectifs

Ce notebook présente la solution complète du laboratoire. Chaque étape suit exactement la même structure que la version étudiante, mais avec le code, les réponses attendues et des explications détaillées.

**Variables du CSV :**
- `latitude`
- `longitude`
- `pp_open`

**Objectifs :**
- charger un fichier CSV ;
- créer une base SQLite ;
- écrire des requêtes SQL simples ;
- convertir les données en objets spatiaux ;
- sélectionner les points dans une région ;
- calculer des statistiques descriptives.


## 2. Importer les bibliothèques

On importe :
- `pandas` pour lire et manipuler le tableau ;
- `sqlite3` pour créer une petite base SQL locale ;
- `geopandas` pour les opérations spatiales ;
- `Point` pour créer une géométrie à partir de coordonnées ;
- `statistics` pour calculer moyenne et écart-type.


In [ ]:
import pandas as pd
import sqlite3
import geopandas as gpd
from shapely.geometry import Point
import statistics


## 3. Charger le fichier CSV

`pd.read_csv()` lit le fichier texte tabulaire et le convertit en DataFrame. Le chemin est relatif au dépôt GitHub/Binder.


In [ ]:
df = pd.read_csv('data/cci_v6_2003226_ppz_takuvik_above_45n_v2.csv')
df.head()


## 4. Explorer les colonnes et le contenu

Cette étape sert à vérifier que les noms de colonnes sont bien ceux attendus avant de construire les requêtes SQL et les objets spatiaux.


In [ ]:
print('Colonnes du fichier :', list(df.columns))
print('Nombre total de lignes :', len(df))
print('\nTypes de variables :')
print(df.dtypes)


## 5. Créer la base SQLite

`sqlite3.connect()` ouvre ou crée un fichier de base de données. `to_sql()` écrit le DataFrame dans une table SQL. `if_exists='replace'` remplace la table si elle existe déjà.


In [ ]:
conn = sqlite3.connect('pp_database.sqlite')
df.to_sql('pp_data', conn, if_exists='replace', index=False)
print('Base de données créée avec succès.')


## 6. Requête SQL 1 : nombre de pixels

`COUNT(*)` compte toutes les lignes de la table.


In [ ]:
query = 'SELECT COUNT(*) AS nb_pixels FROM pp_data;'
pd.read_sql_query(query, conn)


## 7. Requête SQL 2 : minimum et maximum

`MIN()` et `MAX()` résument l’étendue des valeurs de la variable.


In [ ]:
query = '''
SELECT MIN(pp_open) AS pp_min, MAX(pp_open) AS pp_max
FROM pp_data;
'''
pd.read_sql_query(query, conn)


## 8. Requête SQL 3 : moyenne globale

`AVG()` retourne la moyenne arithmétique d’une colonne numérique.


In [ ]:
query = 'SELECT AVG(pp_open) AS moyenne_globale FROM pp_data;'
pd.read_sql_query(query, conn)


## 9. Requête SQL 4 : pixels au-dessus d’un seuil

La clause `WHERE` permet de filtrer les lignes avant le calcul.


In [ ]:
query = 'SELECT COUNT(*) AS nb_pixels_sup_1000 FROM pp_data WHERE pp_open > 1000;'
pd.read_sql_query(query, conn)


## 10. Requête SQL 5 : filtre par latitude

Cette requête sert d’exemple de filtre simple sur une variable numérique.


In [ ]:
query = 'SELECT * FROM pp_data WHERE latitude > 75;'
pd.read_sql_query(query, conn).head()


## 11. Convertir le tableau en GeoDataFrame

On transforme chaque paire de coordonnées `(longitude, latitude)` en point géométrique. Le système `EPSG:4326` correspond à des coordonnées géographiques en degrés.


In [ ]:
geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
gdf.head()


## 12. Charger le shapefile

`gpd.read_file()` lit le shapefile. Les autres fichiers associés (`.shx`, `.dbf`, `.prj`) doivent être présents dans le même dossier.


In [ ]:
region = gpd.read_file('data/WDPA_WDOECM_Feb2025_Public_555637925_shp-polygons.shp')
region.head()


## 13. Vérifier / harmoniser le système de coordonnées

Deux couches doivent être dans le même système de coordonnées pour qu’une jointure spatiale fonctionne correctement. Ici, on convertit le shapefile vers le SCR des points.


In [ ]:
print('SCR des points :', gdf.crs)
print('SCR du shapefile avant conversion :', region.crs)
region = region.to_crs(gdf.crs)
print('SCR du shapefile après conversion :', region.crs)


## 14. Faire la sélection spatiale

La jointure spatiale compare la position des points et des polygones. Avec `predicate='within'`, seuls les points à l’intérieur des polygones sont conservés.


In [ ]:
points_region = gpd.sjoin(gdf, region, predicate='within')
points_region.head()


## 15. Compter les points dans la région

Après la sélection spatiale, `len(points_region)` donne le nombre de points qui satisfont la condition spatiale.


In [ ]:
print('Nombre de pixels dans la région :', len(points_region))


## 16. Calculer les statistiques descriptives

On extrait la colonne d’intérêt sous forme de liste numérique. `dropna()` élimine les valeurs manquantes. Ensuite, on calcule la moyenne, l’écart-type, le minimum et le maximum.


In [ ]:
valeurs = points_region['pp_open'].dropna().tolist()

moyenne = statistics.mean(valeurs)
ecart_type = statistics.stdev(valeurs)
minimum = min(valeurs)
maximum = max(valeurs)

print('Moyenne :', moyenne)
print('Écart-type :', ecart_type)
print('Minimum :', minimum)
print('Maximum :', maximum)


## 17. Interpréter les résultats

Cette partie ne donne pas une seule bonne réponse : elle sert à discuter la signification des statistiques. On peut comparer la moyenne régionale à la moyenne globale, commenter la variabilité à partir de l’écart-type et souligner les limites, par exemple :
- sensibilité à la zone choisie ;
- absence d’analyse temporelle ;
- distribution potentiellement asymétrique ;
- dépendance à la qualité du shapefile et du CSV.


## 18. Fermer la connexion

Fermer la connexion libère proprement la ressource associée au fichier SQLite.


In [ ]:
conn.close()
print('Connexion fermée.')
